# Dictionary Polysemy — Simplex vs. Root+*hata* Minimal Pairs

Sense counts from STDICT and Urimalsaem, filtered by part of speech and disambiguated by homograph group.

In [1]:
import xml.etree.ElementTree as ET
import requests
import pandas as pd
from api_keys_local import STDICT_KEY, WOORIMALSAM_KEY

STDICT_SEARCH_URL = "https://stdict.korean.go.kr/api/search.do"
STDICT_VIEW_URL = "https://stdict.korean.go.kr/api/view.do"
WOORIMALSAM_SEARCH_URL = "https://opendict.korean.go.kr/api/search"

PAIRS = [
    ("헤아리다", "생각하다", "동사"),
    ("부르다", "노래하다", "동사"),
    ("게우다", "토하다", "동사"),
    ("굽다", "요리하다", "동사"),
    ("덥다", "따뜻하다", "형용사"),
]

def _as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

In [2]:
def count_senses_stdict(word, pos, key):
    params = {"key": key, "q": word, "req_type": "json", "method": "exact", "num": 100}
    resp = requests.get(STDICT_SEARCH_URL, params=params, timeout=15)
    resp.raise_for_status()
    items = _as_list(resp.json().get("channel", {}).get("item"))
    target_codes = [it["target_code"] for it in items
                    if it.get("word", "").replace("-", "") == word and it.get("pos") == pos]
    counts = []
    for tc in target_codes:
        vresp = requests.get(STDICT_VIEW_URL,
                              params={"key": key, "method": "target_code", "q": tc, "req_type": "xml"},
                              timeout=15)
        root = ET.fromstring(vresp.text)
        n = sum(len(pi.findall(".//sense_info")) for pi in root.findall(".//pos_info")
                if (pi.findtext("pos") or "").strip() == pos)
        counts.append(n)
    return max(counts) if counts else 0

def count_senses_woorimalsam(word, pos, key):
    params = {"key": key, "q": word, "req_type": "json", "method": "exact", "num": 100}
    resp = requests.get(WOORIMALSAM_SEARCH_URL, params=params, timeout=15)
    resp.raise_for_status()
    items = _as_list(resp.json().get("channel", {}).get("item"))
    counts = []
    for it in items:
        if it.get("word", "").replace("-", "") != word:
            continue
        n = len([s for s in _as_list(it.get("sense")) if s.get("pos") == pos])
        if n:
            counts.append(n)
    return max(counts) if counts else 0

In [3]:
rows = []
for simplex, hada, pos in PAIRS:
    for source, fn, key in [("STDICT", count_senses_stdict, STDICT_KEY),
                             ("Urimalsaem", count_senses_woorimalsam, WOORIMALSAM_KEY)]:
        rows.append([simplex, hada, source, "simplex", fn(simplex, pos, key)])
        rows.append([simplex, hada, source, "hada", fn(hada, pos, key)])

df = pd.DataFrame(rows, columns=["pair_simplex", "pair_hada", "source", "role", "sense_count"])
df.to_csv("dictionary_polysemy_results.csv", index=False, encoding="utf-8-sig")
df

,pair_simplex,pair_hada,source,role,sense_count
0,헤아리다,생각하다,STDICT,simplex,3
1,헤아리다,생각하다,STDICT,hada,7
2,헤아리다,생각하다,Urimalsaem,simplex,3
3,헤아리다,생각하다,Urimalsaem,hada,7
4,부르다,노래하다,STDICT,simplex,10
5,부르다,노래하다,STDICT,hada,5
6,부르다,노래하다,Urimalsaem,simplex,10
7,부르다,노래하다,Urimalsaem,hada,5
8,게우다,토하다,STDICT,simplex,2
9,게우다,토하다,STDICT,hada,3


In [4]:
concept_map = {("헤아리다", "생각하다"): "think", ("부르다", "노래하다"): "sing",
               ("게우다", "토하다"): "vomit", ("굽다", "요리하다"): "cook",
               ("덥다", "따뜻하다"): "warm"}

wide = df.pivot_table(index=["pair_simplex", "pair_hada"], columns="role", values="sense_count", aggfunc="first")
wide.insert(0, "concept", [concept_map[i] for i in wide.index])
wide["direction"] = wide.apply(
    lambda r: "simplex > hada (matches hypothesis)" if r["simplex"] > r["hada"]
    else "hada > simplex (opposite)", axis=1)
wide.set_index("concept")[["simplex", "hada", "direction"]]

role,simplex,hada,direction
concept,,,
vomit,2,3,hada > simplex (opposite)
cook,7,2,simplex > hada (matches hypothesis)
warm,4,2,simplex > hada (matches hypothesis)
sing,10,5,simplex > hada (matches hypothesis)
think,3,7,hada > simplex (opposite)
